<a href="https://colab.research.google.com/github/ha22756/images/blob/main/extract_geometric_features_colab2_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ADT Segmentation and Geometric Feature Extraction (English Version)

This notebook is written in **English** and includes **explanatory text before each Python cell**.

## Goal
This notebook:
1. uploads `roi_exam.mat`,
2. applies **ADT segmentation** to each nodule ROI,
3. extracts **geometric features** from the segmented nodules,
4. saves the results to:
   - `geometric_features_results.csv`
   - `geometric_features_results.mat`
   - `segmentation_preview.png`

## Important note
Some features in the original paper depend on the **full lung field** from the original chest radiograph.  
Because `roi_exam.mat` contains only **local ROI patches**, this notebook computes **local proxy versions** for the location-based features.


## Cell 1 — Install required libraries

This cell installs the Python packages needed in Google Colab:
- `numpy`
- `scipy`
- `matplotlib`
- `scikit-image`
- `pandas`


In [1]:
!pip -q install numpy scipy matplotlib scikit-image pandas


## Cell 2 — Upload the `.mat` file

This cell opens the upload window in Google Colab.  
Please upload your file:

- `roi_exam.mat`

After uploading, the file will be available in the Colab working directory.


In [ ]:
from google.colab import files
uploaded = files.upload()


## Cell 3 — Import libraries

This cell imports all libraries used later in the notebook for:
- loading `.mat` files,
- image processing,
- segmentation,
- geometric measurement,
- saving results,
- plotting preview images.


In [ ]:
from pathlib import Path
import numpy as np
import scipy.io as sio
from scipy import ndimage as ndi
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
from skimage.morphology import disk, opening
from skimage.measure import label, regionprops
import pandas as pd


## Cell 4 — Define the ADT segmentation functions

This cell defines the functions used for **Adaptive Distance-Based Threshold (ADT) segmentation**:
- `create_distance_array`: creates the distance map around the cue point,
- `threshold_matrix`: builds the adaptive threshold surface,
- `extract_component_connected_to_cue`: keeps only the connected component containing the cue,
- `radial_gradient_all`: computes the average radial gradient,
- `adt_segmentation`: applies the full ADT segmentation pipeline to one ROI image.


In [ ]:
def create_distance_array(shape=(87, 87), cue=(43, 43)):
    rows, cols = shape
    yy, xx = np.indices((rows, cols))
    cx, cy = cue
    d = (xx - cx) ** 2 + (yy - cy) ** 2
    return d


def threshold_matrix(d: np.ndarray, rm: float, TD: float, T0: float) -> np.ndarray:
    T = T0 + TD * ((1 - np.exp(-d / (rm**2))) / (1 - np.exp(-1)))
    T = T.astype(float)
    T[d >= rm**2] = np.inf
    return T


def extract_component_connected_to_cue(mask: np.ndarray, cue=(43, 43)) -> np.ndarray:
    structure = np.array([[0, 1, 0],
                          [1, 1, 1],
                          [0, 1, 0]], dtype=int)
    labeled, _ = ndi.label(mask, structure=structure)
    cx, cy = cue
    cue_label = labeled[cy, cx]
    if cue_label == 0:
        return np.zeros_like(mask, dtype=bool)
    return labeled == cue_label


def radial_gradient_all(gradx: np.ndarray, grady: np.ndarray, mask: np.ndarray, cue=(43, 43)):
    gradx = np.asarray(gradx, dtype=float)
    grady = np.asarray(grady, dtype=float)
    mask = np.asarray(mask).astype(bool)

    ys, xs = np.where(mask)
    if len(xs) == 0:
        return -np.inf, np.nan

    grad_vectors = np.column_stack((-gradx[ys, xs], -grady[ys, xs]))
    cx, cy = cue
    radial_vectors = np.column_stack((xs - cx, ys - cy))
    mags = np.sqrt(np.sum(radial_vectors**2, axis=1))
    mags[mags == 0] = 1.0
    radial_unit_vectors = radial_vectors / mags[:, None]

    inner_products = np.sum(grad_vectors * radial_unit_vectors, axis=1)
    return float(np.mean(inner_products)), float(np.std(inner_products))


def adt_segmentation(
    x: np.ndarray,
    lung_mask: np.ndarray,
    cue=(43, 43),
    rm: float = 25.0,
    TD: float = 1.7,
    TO_values: np.ndarray | None = None,
    d: np.ndarray | None = None,
):
    if TO_values is None:
        TO_values = np.arange(-2.0, 2.0001, 0.01)

    x = np.asarray(x, dtype=float)
    lung_mask = np.asarray(lung_mask).astype(bool)

    if d is None:
        d = create_distance_array(x.shape, cue=cue)

    grady, gradx = np.gradient(x)
    avg_scores = np.zeros(len(TO_values), dtype=float)

    for i, t0 in enumerate(TO_values):
        T = threshold_matrix(d, rm, TD, t0)
        Y = x > T
        mask1 = Y & lung_mask
        nodule = ndi.binary_fill_holes(mask1)
        nodule_open = opening(nodule, footprint=disk(1))
        candidate = extract_component_connected_to_cue(nodule_open, cue=cue)
        avg_scores[i], _ = radial_gradient_all(gradx, grady, candidate, cue)

    best_idx = int(np.argmax(avg_scores))
    TO_best = float(TO_values[best_idx])
    T_best = threshold_matrix(d, rm, TD, TO_best)

    Y = x > T_best
    mask1 = Y & lung_mask
    nodule = ndi.binary_fill_holes(mask1)
    nodule_open = opening(nodule, footprint=disk(1))
    mask = extract_component_connected_to_cue(nodule_open, cue=cue)

    return mask.astype(np.uint8), avg_scores, TO_best, T_best


## Cell 5 — Define the geometric feature extraction functions

This cell defines the functions that compute geometric features from the segmented mask.

The extracted features include:
- maximum size,
- area,
- centroid position,
- eccentricity,
- orientation,
- two circularity measures,
- normalized distance to the local lung perimeter.

Because the ROI file is local, the position-based features are computed as **local ROI proxies**.


In [ ]:
def longest_diameter_pixels(mask: np.ndarray) -> float:
    ys, xs = np.where(mask > 0)
    if len(xs) <= 1:
        return 0.0
    pts = np.column_stack((xs, ys)).astype(float)
    return float(pdist(pts).max())


def local_lung_bbox(lung_mask: np.ndarray):
    ys, xs = np.where(lung_mask > 0)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())


def compute_geometric_features(mask: np.ndarray,
                               lung_mask: np.ndarray,
                               pixel_spacing: float = 1.0) -> dict:
    mask = np.asarray(mask).astype(bool)
    lung_mask = np.asarray(lung_mask).astype(bool)

    if not np.any(mask):
        return {
            "size_px": np.nan,
            "size_mm": np.nan,
            "area_px": 0.0,
            "area_mm2": 0.0,
            "x_fraction_local_lung_bbox": np.nan,
            "y_fraction_local_lung_bbox": np.nan,
            "eccentricity": np.nan,
            "orientation_deg": np.nan,
            "circularity1_bbox": np.nan,
            "circularity2_size": np.nan,
            "distance_to_lung_perimeter_norm": np.nan,
            "centroid_x_px": np.nan,
            "centroid_y_px": np.nan,
        }

    labeled = label(mask)
    props = regionprops(labeled.astype(int))
    prop = max(props, key=lambda p: p.area)

    area_px = float(prop.area)
    area_mm2 = area_px * (pixel_spacing ** 2)

    cy, cx = prop.centroid
    centroid_x_px = float(cx)
    centroid_y_px = float(cy)

    size_px = longest_diameter_pixels(mask)
    size_mm = size_px * pixel_spacing

    eccentricity = float(prop.eccentricity)
    orientation_deg = float(np.degrees(prop.orientation))

    minr, minc, maxr, maxc = prop.bbox
    bbox_width = maxc - minc
    bbox_height = maxr - minr
    r_bbox = max(bbox_width, bbox_height) / 2.0
    circularity1_bbox = float(area_px / (np.pi * (r_bbox ** 2))) if r_bbox > 0 else np.nan

    r_size = size_px / 2.0
    circularity2_size = float(area_px / (np.pi * (r_size ** 2))) if r_size > 0 else np.nan

    bbox = local_lung_bbox(lung_mask)
    if bbox is None:
        x_fraction = np.nan
        y_fraction = np.nan
    else:
        xmin, xmax, ymin, ymax = bbox
        lung_w = max(xmax - xmin, 1)
        lung_h = max(ymax - ymin, 1)
        x_fraction = float((cx - xmin) / lung_w)
        y_fraction = float((cy - ymin) / lung_h)

    dt = ndi.distance_transform_edt(lung_mask)
    dt_max = float(dt.max())
    cx_i = int(np.clip(round(cx), 0, dt.shape[1] - 1))
    cy_i = int(np.clip(round(cy), 0, dt.shape[0] - 1))
    dist_norm = float(dt[cy_i, cx_i] / dt_max) if dt_max > 0 else np.nan

    return {
        "size_px": size_px,
        "size_mm": size_mm,
        "area_px": area_px,
        "area_mm2": area_mm2,
        "x_fraction_local_lung_bbox": x_fraction,
        "y_fraction_local_lung_bbox": y_fraction,
        "eccentricity": eccentricity,
        "orientation_deg": orientation_deg,
        "circularity1_bbox": circularity1_bbox,
        "circularity2_size": circularity2_size,
        "distance_to_lung_perimeter_norm": dist_norm,
        "centroid_x_px": centroid_x_px,
        "centroid_y_px": centroid_y_px,
    }


## Cell 6 — Define helper functions for loading data and making a preview image

This cell defines:
- `load_roi_exam`: reads the `roi_exam.mat` file,
- `make_preview`: creates a figure showing the segmentation result for all 8 ROIs.


In [ ]:
def load_roi_exam(mat_path: str | Path):
    data = sio.loadmat(mat_path, simplify_cells=True)
    return data["roi_exam"]


def make_preview(roi_exam, masks, output_path="segmentation_preview.png"):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = np.ravel(axes)

    for i, (item, mask) in enumerate(zip(roi_exam, masks)):
        ax = axes[i]
        x = np.asarray(item["cxr_contrast"], dtype=float)
        ax.imshow(x, cmap="gray")
        ax.contour(mask.astype(float), levels=[0.5], colors=["lime"], linewidths=[1.5])
        ax.set_title(f'{item.get("base_name", "ROI")} #{i+1}', fontsize=9)
        ax.axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


## Cell 7 — Run the full pipeline

This is the main execution cell.

It:
1. loads `roi_exam.mat`,
2. loops over all ROIs,
3. performs ADT segmentation,
4. extracts the geometric features,
5. stores the masks and ARG curves,
6. saves the final results to CSV and MAT files,
7. creates a segmentation preview image.


In [ ]:
mat_path = Path("roi_exam.mat")
if not mat_path.exists():
    raise FileNotFoundError("roi_exam.mat was not found. Please upload it first.")

roi_exam = load_roi_exam(mat_path)

rm = 25.0
TD = 1.7
TO_values = np.arange(-2.0, 2.0001, 0.01)

all_features = []
all_masks = []
all_t0 = []
all_arg_curves = []

for i, item in enumerate(roi_exam, start=1):
    x = np.asarray(item["cxr_contrast"], dtype=float)
    lung_mask = np.asarray(item["lung_mask"]).astype(bool)
    pixel_spacing = float(item.get("pixel_spacing", 1.0))

    truth_cue = item.get("truth_cue", None)
    if truth_cue is not None and len(np.ravel(truth_cue)) >= 2:
        cue_x = int(round(float(np.ravel(truth_cue)[0]) - 1))
        cue_y = int(round(float(np.ravel(truth_cue)[1]) - 1))
        cue = (cue_x, cue_y)
    else:
        cue = (43, 43)

    d = create_distance_array(shape=x.shape, cue=cue)

    mask, avg_scores, T0_best, _ = adt_segmentation(
        x=x,
        lung_mask=lung_mask,
        cue=cue,
        rm=rm,
        TD=TD,
        TO_values=TO_values,
        d=d,
    )

    feats = compute_geometric_features(mask, lung_mask, pixel_spacing=pixel_spacing)
    feats["roi_index"] = i
    feats["base_name"] = str(item.get("base_name", f"roi_{i:02d}"))
    feats["nodule_idx"] = int(item.get("nodule_idx", i))
    feats["truth_size_mm"] = float(item.get("truth_size_mm", np.nan))
    feats["subtlety"] = float(item.get("subtlety", np.nan))
    feats["pixel_spacing"] = pixel_spacing
    feats["T0_best"] = float(T0_best)

    all_features.append(feats)
    all_masks.append(mask.astype(np.uint8))
    all_t0.append(float(T0_best))
    all_arg_curves.append(avg_scores.astype(np.float32))

df = pd.DataFrame(all_features)

masks_3d = np.stack(all_masks, axis=2).astype(np.uint8)
arg_curves = np.stack(all_arg_curves, axis=0).astype(np.float32)

df.to_csv("geometric_features_results.csv", index=False)

mat_out = {
    "feature_names": np.array(df.columns.tolist(), dtype=object),
    "features_table": df.to_dict(orient="list"),
    "features_matrix": df.select_dtypes(include=[np.number]).to_numpy(dtype=float),
    "numeric_feature_names": np.array(df.select_dtypes(include=[np.number]).columns.tolist(), dtype=object),
    "segmentation_masks": masks_3d,
    "T0_best": np.array(all_t0, dtype=float),
    "ARG_curves": arg_curves,
    "TO_values": TO_values.astype(float),
}
sio.savemat("geometric_features_results.mat", mat_out, do_compression=True)

make_preview(roi_exam, all_masks, output_path="segmentation_preview.png")

print("Done.")
print("Saved: geometric_features_results.csv")
print("Saved: geometric_features_results.mat")
print("Saved: segmentation_preview.png")
print()
print("Important note:")
print("x_fraction_local_lung_bbox and y_fraction_local_lung_bbox are local ROI proxies.")
print("They are not the exact paper-level full-lung X-Fraction and Y-Fraction.")


## Cell 8 — View the extracted feature table

This cell reads the generated CSV file and displays the feature table inside Colab.


In [ ]:
df_results = pd.read_csv("geometric_features_results.csv")
df_results


## Cell 9 — Display the segmentation preview image

This cell shows the image that overlays the segmentation mask on each ROI.


In [ ]:
from IPython.display import Image, display
display(Image("segmentation_preview.png"))


## Cell 10 — Download the result files

This final cell downloads the output files from Colab to your computer:
- `geometric_features_results.csv`
- `geometric_features_results.mat`
- `segmentation_preview.png`


In [ ]:
from google.colab import files
files.download("geometric_features_results.csv")
files.download("geometric_features_results.mat")
files.download("segmentation_preview.png")
